In [0]:
from datetime import datetime

try:

    start_time = datetime.now()

    df = spark.table(
        "workspace.consolidated_upi.daily_txn_summary"
    )

    record_count = df.count()

    end_time = datetime.now()

    duration = (
        end_time - start_time
    ).total_seconds()

    status = "SUCCESS"

    error_message = None

except Exception as e:

    end_time = datetime.now()

    duration = (
        end_time - start_time
    ).total_seconds()

    status = "FAILED"

    error_message = str(e)

    record_count = 0

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DoubleType, LongType, DateType
)
from datetime import datetime

schema = StructType([
    StructField("job_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("execution_date", DateType(), True)
])

log_df = spark.createDataFrame([
    (
        "Daily_Transaction_Summary_Pipeline",
        "Gold",
        start_time,
        end_time,
        float(duration),
        status,
        int(record_count),
        error_message,   
        datetime.now().date()
    )
], schema=schema)

In [0]:
log_df.write.mode("append").saveAsTable(
    "workspace.monitoring.monitor_job_execution"
)

In [0]:
display(
    spark.table(
        "workspace.monitoring.monitor_job_execution"
    )
)

job_name,layer,start_time,end_time,duration_seconds,status,records_processed,error_message,execution_date
Daily_Transaction_Summary_Pipeline,Gold,2026-05-12T10:59:20.327Z,2026-05-12T10:59:21.339Z,1.011903,SUCCESS,2040,null,2026-05-12
Daily_Transaction_Summary_Pipeline,Gold,2026-05-12T11:08:25.497Z,2026-05-12T11:08:25.872Z,0.375006,SUCCESS,2040,null,2026-05-12
Daily_Transaction_Summary_Pipeline,Gold,2026-05-12T11:14:55.386Z,2026-05-12T11:14:55.713Z,0.327288,SUCCESS,2040,null,2026-05-12


In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_job_execution
ORDER BY start_time DESC;

job_name,layer,start_time,end_time,duration_seconds,status,records_processed,error_message,execution_date
Daily_Transaction_Summary_Pipeline,Gold,2026-05-12T11:14:55.386Z,2026-05-12T11:14:55.713Z,0.327288,SUCCESS,2040,null,2026-05-12
Daily_Transaction_Summary_Pipeline,Gold,2026-05-12T11:08:25.497Z,2026-05-12T11:08:25.872Z,0.375006,SUCCESS,2040,null,2026-05-12
Daily_Transaction_Summary_Pipeline,Gold,2026-05-12T10:59:20.327Z,2026-05-12T10:59:21.339Z,1.011903,SUCCESS,2040,null,2026-05-12


#### monitor_reconciliation

In [0]:
from datetime import datetime
source_table = "workspace.raw_upi.transactions_bronze"
target_table = "workspace.refined_upi.transactions_silver"
source_count = spark.table(source_table).count()
target_count = spark.table(target_table).count()

In [0]:
variance = source_count - target_count
if variance == 0:
    recon_status = "MATCHED"
else:
    recon_status = "MISMATCH"

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("source_count", LongType(), True),
    StructField("target_count", LongType(), True),
    StructField("variance", LongType(), True),
    StructField("recon_status", StringType(), True),
    StructField("check_time", TimestampType(), True)
])

recon_df = spark.createDataFrame([
    (
        "Finance_Pipeline",
        source_table,
        target_table,
        int(source_count),
        int(target_count),
        int(variance),
        recon_status,
        datetime.now()
    )
], schema=schema)

In [0]:
recon_df.write.mode("append").saveAsTable(
    "workspace.monitoring.monitor_reconciliation"
)

In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_reconciliation;

pipeline_name,source_table,target_table,source_count,target_count,variance,recon_status,check_time
Finance_Pipeline,workspace.raw_upi.transactions_bronze,workspace.refined_upi.transactions_silver,3000,3000,0,MATCHED,2026-05-12T13:05:17.421Z


#### monitor_dq_metrics

In [0]:
from pyspark.sql.functions import col
from datetime import datetime

df = spark.table(
    "workspace.refined_upi.transactions_silver"
)

In [0]:
display(df)

transaction_id,transaction_timestamp,sender_vpa,receiver_vpa,receiver_merchant_id,upi_app,transaction_type,amount,status,failure_reason,bank_reference_number,sender_bank,receiver_bank,state,is_repeat_txn,_ingest_timestamp,_source_system,_batch_id,_file_name,transaction_timestamp_ist,is_valid_vpa,transaction_hour,transaction_day_of_week,is_weekend,amount_bucket,merchant_id,merchant_category,merchant_state,fraud_rule,risk_score,is_fraud
UPI00000008,2024-02-26T02:56:28.000Z,user63652@kotak,user96273@bob,null,GPay,P2P,44.14,SUCCESS,null,BNK987253879,Kotak,BOB,Telangana,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-02-26T08:26:28.000Z,true,8,Mon,false,0-500,null,null,null,OTP bypass attempt,0.97,true
UPI00000013,2024-03-30T12:11:52.000Z,user74765@sbi,merchant2182@hdfc,MER00471,BHIM,P2M,null,FAILED,Bank server timeout,BNK460509792,SBI,HDFC,Gujarat,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-03-30T17:41:52.000Z,true,17,Sat,true,10000+,MER00471,Transportation,Telangana,Amount anomaly � 10x avg ticket size,0.58,true
UPI00000026,2024-03-04T13:35:56.000Z,user52737@sbi,merchant2678@axis,MER00066,GPay,Bill Payment,453.11,SUCCESS,null,BNK390963800,SBI,Axis,Telangana,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-03-04T19:05:56.000Z,true,19,Mon,false,0-500,MER00066,Sporting Goods,Telangana,OTP bypass attempt,0.5,true
UPI00000038,2024-03-11T16:34:16.000Z,user95357@yesbank,merchant2086@axis,MER00031,PhonePe,P2M,266.58,SUCCESS,null,BNK636642753,Yes Bank,Axis,Rajasthan,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-03-11T22:04:16.000Z,true,22,Mon,false,0-500,MER00031,Medical and Dental,Gujarat,High velocity � >5 txn in 10 min,0.88,true
UPI00000078,2024-01-10T06:56:53.000Z,user89438@hdfc,merchant1651@sbi,MER00497,Amazon Pay,P2M,null,PENDING,null,null,HDFC,SBI,Tamil Nadu,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-01-10T12:26:53.000Z,true,12,Wed,false,10000+,MER00497,Sporting Goods,Tamil Nadu,Cross-border attempt,0.51,true
UPI00000097,2024-02-10T23:13:16.000Z,user57852@pnb,merchant9218@axis,MER00497,GPay,P2M,null,FAILED,Bank server timeout,BNK547740506,PNB,Axis,West Bengal,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-02-11T04:43:16.000Z,true,4,Sun,true,10000+,MER00497,Sporting Goods,Tamil Nadu,High velocity � >5 txn in 10 min,0.92,true
UPI00000104,2024-01-02T05:51:19.000Z,user55562@yesbank,merchant6983@icici,MER00446,BHIM,Bill Payment,452.16,SUCCESS,null,BNK604001318,Yes Bank,ICICI,Uttar Pradesh,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-01-02T11:21:19.000Z,true,11,Tue,false,0-500,MER00446,Retail Stores,West Bengal,Blacklisted VPA,0.62,true
UPI00000111,2024-01-25T03:00:32.000Z,user39036@kotak,user17040@sbi,null,Amazon Pay,P2P,null,PENDING,null,null,Kotak,SBI,Tamil Nadu,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-01-25T08:30:32.000Z,true,8,Thu,false,10000+,null,null,null,Amount anomaly � 10x avg ticket size,0.55,true
UPI00000123,2024-03-10T09:35:15.000Z,user36445@sbi,merchant7738@hdfc,MER00009,GPay,P2M,32.41,PENDING,null,null,SBI,HDFC,Rajasthan,null,2026-05-06T04:49:39.821Z,upi_system,batch_001,/Volumes/workspace/raw_upi/volumes/transactions/transactions.csv,2024-03-10T15:05:15.000Z,true,15,Sun,true,0-500,MER00009,Drug Stores / Pharmacies,West Bengal,Mule account pattern,0.56,true
UPI00000126,2024-03-09T03:36:36.000Z,user86432@pnb,user59490@hdfc,null,PhonePe,P2P,179.72,SUCCESS,null,BNK780581628,PNB,HDFC,Pune,null,2026-05-06T04:49:39.821Z,upi_system,bat

In [0]:
total_count = df.count()

In [0]:
null_count = df.filter(
    col("receiver_merchant_id").isNull()
).count()
null_percentage = (
    null_count / total_count
) * 100

In [0]:
threshold = 5.0

if null_percentage > threshold:
    dq_status = "FAILED"
else:
    dq_status = "PASSED"

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("metric_name", StringType(), True),
    StructField("metric_value", DoubleType(), True),
    StructField("threshold_value", DoubleType(), True),
    StructField("dq_status", StringType(), True),
    StructField("check_time", TimestampType(), True)
])

dq_df = spark.createDataFrame([
    (
        "Finance_Pipeline",
        "workspace.refined_upi.transactions_silver",
        "null_percentage_transaction_id",
        float(null_percentage),
        float(threshold),
        dq_status,
        datetime.now()
    )
], schema=schema)

In [0]:
dq_df.write.mode("append").saveAsTable(
    "workspace.monitoring.monitor_dq_metrics"
)

In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_dq_metrics;

pipeline_name,table_name,metric_name,metric_value,threshold_value,dq_status,check_time
Finance_Pipeline,workspace.refined_upi.transactions_silver,null_percentage_transaction_id,41.43333333333333,5.0,FAILED,2026-05-12T13:16:27.555Z
Finance_Pipeline,workspace.refined_upi.transactions_silver,null_percentage_transaction_id,0.0,5.0,PASSED,2026-05-12T13:14:34.065Z


###### Duplicate_record_monitoring

In [0]:
total_count = df.count()

distinct_count = df.select(
    "transaction_id"
).distinct().count()

duplicate_count = total_count - distinct_count

In [0]:
duplicate_percentage = (
    duplicate_count / total_count
) * 100
duplicate_threshold = 2.0

if duplicate_percentage > duplicate_threshold:
    duplicate_status = "FAILED"
else:
    duplicate_status = "PASSED"

In [0]:
duplicate_dq_df = spark.createDataFrame([
    (
        "Finance_Pipeline",
        "workspace.refined_upi.transactions_silver",
        "duplicate_percentage_transaction_id",
        float(duplicate_percentage),
        float(duplicate_threshold),
        duplicate_status,
        datetime.now()
    )
], schema=schema)

In [0]:
duplicate_dq_df.write.mode("append").saveAsTable(
    "workspace.monitoring.monitor_dq_metrics"
)

In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_dq_metrics;

pipeline_name,table_name,metric_name,metric_value,threshold_value,dq_status,check_time
Finance_Pipeline,workspace.refined_upi.transactions_silver,duplicate_percentage_transaction_id,0.0,2.0,PASSED,2026-05-12T13:19:49.185Z
Finance_Pipeline,workspace.refined_upi.transactions_silver,null_percentage_transaction_id,41.43333333333333,5.0,FAILED,2026-05-12T13:16:27.555Z
Finance_Pipeline,workspace.refined_upi.transactions_silver,null_percentage_transaction_id,0.0,5.0,PASSED,2026-05-12T13:14:34.065Z


In [0]:
if dq_status == "FAILED":

    alert_type = "DQ_ALERT"

    pipeline_name = "Finance_Pipeline"

    severity = "HIGH"

    alert_message = (
        "NULL percentage exceeded threshold "
        "for receiver_merchant_id"
    )

    alert_status = "ACTIVE"

In [0]:
from pyspark.sql.types import *
from datetime import datetime

alert_schema = StructType([
    StructField("alert_type", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("alert_message", StringType(), True),
    StructField("alert_status", StringType(), True),
    StructField("created_time", TimestampType(), True)
])

alert_df = spark.createDataFrame([
    (
        alert_type,
        pipeline_name,
        severity,
        alert_message,
        alert_status,
        datetime.now()
    )
], schema=alert_schema)

In [0]:
alert_df.write.mode("append").saveAsTable(
    "workspace.monitoring.monitor_alerts"
)

In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_alerts;

alert_type,pipeline_name,severity,alert_message,alert_status,created_time
DQ_ALERT,Finance_Pipeline,HIGH,NULL percentage exceeded threshold for receiver_merchant_id,ACTIVE,2026-05-12T13:39:44.277Z


In [0]:
from datetime import datetime
from pyspark.sql.types import *

try:

    start_time = datetime.now()

   
    df = spark.table(
        "workspace.consolidated_upi.daily_txn_summary"
    )

    record_count = df.count()

    status = "SUCCESS"

    error_message = None

except Exception as e:

    status = "FAILED"

    error_message = str(e)

    record_count = 0

    exception_schema = StructType([
        StructField("pipeline_name", StringType(), True),
        StructField("stage_name", StringType(), True),
        StructField("error_message", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("exception_time", TimestampType(), True)
    ])

    exception_df = spark.createDataFrame([
        (
            "Finance_Pipeline",
            "Gold Layer",
            str(e),
            "CRITICAL",
            datetime.now()
        )
    ], schema=exception_schema)

    exception_df.write.mode("append").saveAsTable(
        "workspace.monitoring.monitor_exceptions"
    )

In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_exceptions;

pipeline_name stage_name error_message severity exception_time Finance_Pipeline Gold Layer [TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`consolidated_upi`.`fake_table` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'Aggregate [unresolvedalias(count(1))]
+- 'UnresolvedRelation [workspace, consolidated_upi, fake_table], [], false


JVM stacktrace:
org.apache.spark.sql.catalyst.ExtendedAnalysisException
	at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:94)
	at org.apache.spark.sql.catalyst.analysis.RelationResolution$.throwTableNotFound(RelationResolution.scala:1126)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2(CheckAnalysis.scala:355)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$2$adapted(CheckAnalysis.scala:324)
	at org.apache.spark.sql.catalyst.trees.TreeNode.foreachUp(TreeNode.scala:371)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1(TreeNode.scala:370)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$foreachUp$1$adapted(TreeNode.scala:370)
	at scala.collection.immutable.Vector.foreach(Vector.scala:2125)
	at org.apache.spark.sql.catalyst.trees.TreeNode.foreachUp(TreeNode.scala:370)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis0(CheckAnalysis.scala:324)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis0$(CheckAnalysis.scala:295)
	at org.apache.spark.sql.catalyst.analysis.Analyzer.checkAnalysis0(Analyzer.scala:554)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:280)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis(CheckAnalysis.scala:267)
	at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis$(CheckAnalysis.scala:263)
	at org.apache.spark.sql.catalyst.analysis.Analyzer.checkAnalysis(Analyzer.scala:554)
	at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.$anonfun$resolveInFixedPoint$1(HybridAnalyzer.scala:414)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withTracker(QueryPlanningTracker.scala:266)
	at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.resolveInFixedPoint(HybridAnalyzer.scala:414)
	at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.$anonfun$apply$1(HybridAnalyzer.scala:97)
	at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.withTrackedAnalyzerBridgeState(HybridAnalyzer.scala:134)
	at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.apply(HybridAnalyzer.scala:90)
	at org.apache.spark.sql.catalyst.analysis.Analyzer.$anonfun$executeAndCheck$2(Analyzer.scala:620)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper$.markInAnalyzer(AnalysisHelper.scala:425)
	at org.apache.spark.sql.catalyst.analysis.Analyzer.$anonfun$executeAndCheck$1(Analyzer.scala:620)
	at com.databricks.sql.unity.SAMSnapshotHelper$.visitPlansDuringAnalysis(SAMSnapshotHelper.scala:41)
	at org.apache.spark.sql.catalyst.analysis.Analyzer.executeAndCheck(Analyzer.scala:609)
	at org.apache.spark.sql.execution.QueryExecution.$anonfun$lazyAnalyzed$3(QueryExecution.scala:575)
	at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114)
	at com.databricks.spark.

In [0]:
%sql
SELECT *
FROM workspace.monitoring.pipeline_mapping;

pipeline_name,source_table,target_table,source_layer,target_layer,dashboard_name
Finance_Pipeline,workspace.raw_upi.transactions_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Finance_Dashboard
Risk_Pipeline,workspace.raw_upi.fraud_flags_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Risk_Dashboard
Merchant_Pipeline,workspace.raw_upi.merchants_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Product_Dashboard
Settlement_Pipeline,workspace.raw_upi.bank_settlements_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Finance_Dashboard
Daily_Transaction_Summary_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.daily_txn_summary,Silver,Gold,Finance_Dashboard
Fraud_Analytics_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.fraud_analytics,Silver,Gold,Risk_Dashboard
Merchant_Performance_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.merchant_performance,Silver,Gold,Product_Dashboard
Reconciliation_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.settlement_reconciliation,Silver,Gold,Finance_Dashboard


In [0]:
mapping_df = spark.table(
    "workspace.monitoring.pipeline_mapping"
)

display(mapping_df)

pipeline_name,source_table,target_table,source_layer,target_layer,dashboard_name
Finance_Pipeline,workspace.raw_upi.transactions_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Finance_Dashboard
Risk_Pipeline,workspace.raw_upi.fraud_flags_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Risk_Dashboard
Merchant_Pipeline,workspace.raw_upi.merchants_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Product_Dashboard
Settlement_Pipeline,workspace.raw_upi.bank_settlements_bronze,workspace.refined_upi.transactions_silver,Bronze,Silver,Finance_Dashboard
Daily_Transaction_Summary_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.daily_txn_summary,Silver,Gold,Finance_Dashboard
Fraud_Analytics_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.fraud_analytics,Silver,Gold,Risk_Dashboard
Merchant_Performance_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.merchant_performance,Silver,Gold,Product_Dashboard
Reconciliation_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.settlement_reconciliation,Silver,Gold,Finance_Dashboard


In [0]:
pipeline_rows = mapping_df.collect()

In [0]:
for row in pipeline_rows:

    pipeline_name = row["pipeline_name"]

    source_table = row["source_table"]

    target_table = row["target_table"]

    print(pipeline_name)

    print(source_table)

    print(target_table)

Finance_Pipeline
workspace.raw_upi.transactions_bronze
workspace.refined_upi.transactions_silver
Risk_Pipeline
workspace.raw_upi.fraud_flags_bronze
workspace.refined_upi.transactions_silver
Merchant_Pipeline
workspace.raw_upi.merchants_bronze
workspace.refined_upi.transactions_silver
Settlement_Pipeline
workspace.raw_upi.bank_settlements_bronze
workspace.refined_upi.transactions_silver
Daily_Transaction_Summary_Pipeline
workspace.refined_upi.transactions_silver
workspace.consolidated_upi.daily_txn_summary
Fraud_Analytics_Pipeline
workspace.refined_upi.transactions_silver
workspace.consolidated_upi.fraud_analytics
Merchant_Performance_Pipeline
workspace.refined_upi.transactions_silver
workspace.consolidated_upi.merchant_performance
Reconciliation_Pipeline
workspace.refined_upi.transactions_silver
workspace.consolidated_upi.settlement_reconciliation


In [0]:
from datetime import datetime

for row in pipeline_rows:

    pipeline_name = row["pipeline_name"]

    source_table = row["source_table"]

    target_table = row["target_table"]

    source_count = spark.table(
        source_table
    ).count()

    target_count = spark.table(
        target_table
    ).count()

    variance = source_count - target_count

    if variance == 0:
        recon_status = "MATCHED"
    else:
        recon_status = "MISMATCH"

    recon_df = spark.createDataFrame([
        (
            pipeline_name,
            source_table,
            target_table,
            source_count,
            target_count,
            variance,
            recon_status,
            datetime.now()
        )
    ], [
        "pipeline_name",
        "source_table",
        "target_table",
        "source_count",
        "target_count",
        "variance",
        "recon_status",
        "check_time"
    ])

    recon_df.write.mode("append").saveAsTable(
        "workspace.monitoring.monitor_reconciliation"
    )

In [0]:
%sql
SELECT *
FROM workspace.monitoring.monitor_reconciliation
ORDER BY check_time DESC;

pipeline_name,source_table,target_table,source_count,target_count,variance,recon_status,check_time
Reconciliation_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.settlement_reconciliation,3000,3000,0,MATCHED,2026-05-12T14:10:56.317Z
Merchant_Performance_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.merchant_performance,3000,573,2427,MISMATCH,2026-05-12T14:10:54.262Z
Fraud_Analytics_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.fraud_analytics,3000,1255,1745,MISMATCH,2026-05-12T14:10:52.446Z
Daily_Transaction_Summary_Pipeline,workspace.refined_upi.transactions_silver,workspace.consolidated_upi.daily_txn_summary,3000,2040,960,MISMATCH,2026-05-12T14:10:50.517Z
Settlement_Pipeline,workspace.raw_upi.bank_settlements_bronze,workspace.refined_upi.transactions_silver,1500,3000,-1500,MISMATCH,2026-05-12T14:10:48.952Z
Merchant_Pipeline,workspace.raw_upi.merchants_bronze,workspace.refined_upi.transactions_silver,601,3000,-2399,MISMATCH,2026-05-12T14:10:47.063Z
Risk_Pipeline,workspace.raw_upi.fraud_flags_bronze,workspace.refined_upi.transactions_silver,480,3000,-2520,MISMATCH,2026-05-12T14:10:45.045Z
Finance_Pipeline,workspace.raw_upi.transactions_bronze,workspace.refined_upi.transactions_silver,3000,3000,0,MATCHED,2026-05-12T14:10:42.704Z
Finance_Pipeline,workspace.raw_upi.transactions_bronze,workspace.refined_upi.transactions_silver,3000,3000,0,MATCHED,2026-05-12T13:05:17.421Z


#### gold_pipeline_health

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.monitoring.gold_pipeline_health AS

SELECT

    job_name,

    COUNT(*) as total_runs,

    SUM(
        CASE WHEN status = 'SUCCESS'
        THEN 1 ELSE 0 END
    ) as success_runs,

    SUM(
        CASE WHEN status = 'FAILED'
        THEN 1 ELSE 0 END
    ) as failed_runs

FROM workspace.monitoring.monitor_job_execution

GROUP BY job_name;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM workspace.monitoring.gold_pipeline_health;

job_name,total_runs,success_runs,failed_runs
Daily_Transaction_Summary_Pipeline,3,3,0


#### gold_dq_summary

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.monitoring.gold_dq_summary AS

SELECT

    pipeline_name,

    metric_name,

    AVG(metric_value) as avg_metric,

    MAX(metric_value) as max_metric,

    MIN(metric_value) as min_metric

FROM workspace.monitoring.monitor_dq_metrics

GROUP BY pipeline_name, metric_name;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM workspace.monitoring.gold_dq_summary ;

pipeline_name,metric_name,avg_metric,max_metric,min_metric
Finance_Pipeline,duplicate_percentage_transaction_id,0.0,0.0,0.0
Finance_Pipeline,null_percentage_transaction_id,20.716666666666665,41.43333333333333,0.0


#### gold_reconciliation_summary


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.monitoring.gold_reconciliation_summary AS

SELECT

    pipeline_name,

    COUNT(*) as total_checks,

    SUM(
        CASE WHEN recon_status = 'MATCHED'
        THEN 1 ELSE 0 END
    ) as matched_count,

    SUM(
        CASE WHEN recon_status = 'MISMATCH'
        THEN 1 ELSE 0 END
    ) as mismatch_count

FROM workspace.monitoring.monitor_reconciliation

GROUP BY pipeline_name;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from  workspace.monitoring.gold_reconciliation_summary

pipeline_name,total_checks,matched_count,mismatch_count
Daily_Transaction_Summary_Pipeline,1,0,1
Merchant_Performance_Pipeline,1,0,1
Reconciliation_Pipeline,1,1,0
Fraud_Analytics_Pipeline,1,0,1
Settlement_Pipeline,1,0,1
Finance_Pipeline,2,2,0
Merchant_Pipeline,1,0,1
Risk_Pipeline,1,0,1


#### gold_alert_summary

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.monitoring.gold_alert_summary AS

SELECT

    severity,

    COUNT(*) as alert_count

FROM workspace.monitoring.monitor_alerts

GROUP BY severity;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from  workspace.monitoring.gold_alert_summary

severity,alert_count
HIGH,1


  File <command-5713490971182465>, line 1
    SHOW TABLES IN workspace.monitoring;
         ^
SyntaxError: invalid syntax
